# 第 1 天：职位帖分析（Groq）

## 练习目标（理念）

抓取招聘页面文本，并用 **Groq** 上的 Llama 模型做结构化职位分析与投递建议。

- **输入**：职位页 URL
- **输出**：Job Title / Skills / Experience / Salary clues / Apply-or-Skip

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `requests` + BeautifulSoup 去噪 |
| Chat Completions | `client.chat.completions.create(...)` |
| system / user 分工 | system 定分析规则，user 放职位正文 |
| API Key 环境变量 | `GROQ_API_KEY`（通常以 `gsk_` 开头） |

## 怎么跑

1. 依次运行单元格；先确认密钥检查打印 `Groq API key looks good!`
2. `.env` 中配置 `GROQ_API_KEY`
3. 最后一格填入 `url`，再跑抓取与分析

> 说明：分析函数使用 `client`；若上文未创建 `Groq` 客户端，需按作者意图补齐（此处不改逻辑）。


In [ ]:
# ========== 导入：HTTP 抓取 + HTML 解析 + Groq SDK ==========

# os：读环境变量里的 API Key
import os
# requests：HTTP 客户端，下载职位页
import requests
# BeautifulSoup：解析 HTML，抽取可见文本
from bs4 import BeautifulSoup
# load_dotenv：从 .env 加载密钥到进程环境
from dotenv import load_dotenv
# Groq：云端推理客户端（接口形态接近 OpenAI Chat Completions）
from groq import Groq


In [ ]:
# ========== 环境检查：GROQ_API_KEY 是否可读、格式是否像 Groq 密钥 ==========

# override=True：.env 中的值覆盖已存在的同名环境变量
load_dotenv(override=True)
# 取出密钥；标识符与环境变量名保持英文原样
api_key = os.getenv('GROQ_API_KEY')

# 分层诊断：缺省 / 前缀 / 空白 —— 错误提示字符串保持英文
if not api_key:
    print("No API key found")
elif not api_key.startswith("gsk_"):
    print("Invalid Groq API key format")
elif api_key.strip() != api_key:
    print("API key has extra spaces")
else:
    print("Groq API key looks good!")


## 抓取器（Scraper）

`fetch_job_text` 把职位 URL 变成干净纯文本：去掉脚本与样式，再折叠空白，方便塞进 user prompt。


In [ ]:
# ========== 抓取函数：URL → 纯文本职位帖 ==========

def fetch_job_text(url):
    # 抓取职位页纯文本
    # 带 User-Agent，降低被站点直接拒掉的概率
    headers = {"User-Agent": "Mozilla/5.0"}

    # 同步 GET；失败时会抛异常（本练习未包 try/except，保持原样）
    response = requests.get(url, headers=headers)
    # html.parser：标准库可用的解析器后端
    soup = BeautifulSoup(response.text, "html.parser")

    # 去掉脚本/样式等噪音，避免把 JS/CSS 喂给模型
    for tag in soup(["script", "style"]):
        tag.decompose()

    # 抽取文本；块之间用空格分隔
    text = soup.get_text(separator=" ")

    # 归一化空白：多个空格/换行 → 单个空格
    return " ".join(text.split())


In [ ]:
# ========== system prompt：专家角色 + 固定返回结构（英文指令不翻译）==========

system_prompt = """
You are an expert AI Job Analyst.

Your job:
- Analyze job posts clearly and structured
- Extract useful insights
- Be concise and practical

Return format:
1. Job Title
2. Required Skills
3. Experience Level
4. Salary clues (if any)
5. Recommendation (Apply or Skip with reason)

Rules:
- Be direct
- No fluff
- Use bullet points when needed
"""


In [ ]:
# ========== user prompt 前缀：后面会与 job_text 字符串拼接 ==========

# 模板含 {job_text}，但调用处未 .format()；占位符会原样进入模型输入——保持原逻辑
user_prompt = """Analyze this job post: {job_text}"""


In [ ]:
# ========== 调用 Groq Llama：一次 Chat Completions 拿到分析正文 ==========

def analyze_job(job_text):
    # 调用模型分析职位帖
    # model id 必须与 Groq 控制台/文档中的可用名称一致
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            # user_prompt + job_text：前缀说明 + 职位正文
            {"role": "user", "content": user_prompt + (job_text)}
        ]
    )

    # 取第一条 completion 的 message.content
    return response.choices[0].message.content


In [ ]:
# ========== 端到端：填 URL → 抓取 → AI 分析 → 打印结果 ==========

# 在此填入真实招聘页链接（空字符串会导致抓取失败）
url = ""
print("\nScraping job post...\n")
job_text = fetch_job_text(url)

print("Analyzing with AI...\n")
# 把清洗后的文本交给 analyze_job
result = analyze_job(job_text)

print(result)
